## Cell 1: Imports & Configs

In [1]:

from __future__ import annotations

from pathlib import Path

import numpy as np
from joblib import Parallel, delayed
from skimage.color import rgb2gray
from skimage.feature import hog
from skimage.io import imread
from skimage.transform import resize
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from tqdm import tqdm

import joblib
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

SEED = 42
IMAGE_SIZE = (128, 128)
HOG_PARAMS = {
    "orientations": 9,
    "pixels_per_cell": (8, 8),
    "cells_per_block": (2, 2),
    "block_norm": "L2-Hys",
}
VAL_SIZE = 0.2
N_JOBS = 8
OPTUNA_TRIALS = 30
OPTUNA_JOBS = 4


e:\Nam_3_HK2\ThiGiac\DoAn\demo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 2: Helper Functions

In [2]:

def resolve_data_dir() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        candidate = parent / "data"
        if (candidate / "train").is_dir() and (candidate / "test").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/train and data/test")

def collect_paths(split_dir: Path) -> tuple[list[Path], np.ndarray, list[str]]:
    labels = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    if not labels:
        raise RuntimeError(f"No class folders found in {split_dir}")

    paths: list[Path] = []
    y: list[str] = []
    for label in labels:
        for path in sorted((split_dir / label).glob("*.*")):
            if path.is_file():
                paths.append(path)
                y.append(label)

    if not paths:
        raise RuntimeError(f"No images found in {split_dir}")

    return paths, np.array(y), labels

def compute_features(
    paths: list[Path],
    labels: np.ndarray,
    image_size: tuple[int, int],
    hog_params: dict,
    cache_file: Path | None,
    n_jobs: int,
) -> tuple[np.ndarray, np.ndarray]:
    if cache_file is not None and cache_file.exists():
        data = np.load(cache_file)
        return data["X"], data["y"]

    def extract(path: Path) -> np.ndarray | None:
        try:
            img = imread(path)
            gray = rgb2gray(img) if img.ndim == 3 else img
            gray = resize(gray, image_size, anti_aliasing=True)
            return hog(gray, **hog_params)
        except Exception:
            return None

    desc = f"HOG {cache_file.stem}" if cache_file is not None else "HOG"
    features = Parallel(n_jobs=n_jobs)(
        delayed(extract)(p) for p in tqdm(paths, desc=desc, total=len(paths))
    )

    kept_feats: list[np.ndarray] = []
    kept_labels: list[str] = []
    skipped = 0
    for feat, label in zip(features, labels):
        if feat is None:
            skipped += 1
            continue
        kept_feats.append(feat)
        kept_labels.append(label)

    if not kept_feats:
        raise RuntimeError("No features extracted; check input images.")

    X = np.vstack(kept_feats)
    y = np.array(kept_labels)

    if cache_file is not None:
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(cache_file, X=X, y=y)

    if skipped:
        print(f"Skipped {skipped} unreadable files.")

    return X, y


## Cell 3: Data Preparation

In [3]:

DATA_DIR = resolve_data_dir()
MODELS_DIR = DATA_DIR.parent / "models"
CACHE_DIR = DATA_DIR.parent / "cache"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_paths, train_labels, train_classes = collect_paths(DATA_DIR / "train")
test_paths, test_labels, test_classes = collect_paths(DATA_DIR / "test")

if set(train_classes) != set(test_classes):
    print("Warning: train/test class folders differ; using union of labels.")
all_classes = sorted(set(train_classes) | set(test_classes))

train_paths, val_paths, y_train, y_val = train_test_split(
    train_paths,
    train_labels,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_labels,
)


## Cell 4: Feature Extraction & Label Encoding

In [4]:

cache_tag = (
    f"hog_svm_8x2_{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}_"
    f"val{int(VAL_SIZE * 100)}_seed{SEED}"
)
X_train, y_train = compute_features(
    train_paths, y_train, IMAGE_SIZE, HOG_PARAMS, CACHE_DIR / f"{cache_tag}_train.npz", N_JOBS
)
X_val, y_val = compute_features(
    val_paths, y_val, IMAGE_SIZE, HOG_PARAMS, CACHE_DIR / f"{cache_tag}_val.npz", N_JOBS
)
X_test, y_test = compute_features(
    test_paths, test_labels, IMAGE_SIZE, HOG_PARAMS, CACHE_DIR / f"{cache_tag}_test.npz", N_JOBS
)

le = LabelEncoder()
le.fit(all_classes)
y_train_enc = le.transform(y_train)
y_val_enc = le.transform(y_val)
y_test_enc = le.transform(y_test)


## Cell 5: Hyperparameter Tuning

In [5]:

def objective(trial: optuna.Trial) -> float:
    params = {
        "C": trial.suggest_float("C", 1e-2, 1e2, log=True),
        "kernel": trial.suggest_categorical("kernel", ["linear", "rbf"]),
        "gamma": trial.suggest_categorical("gamma", ["scale", "auto"]),
        "random_state": SEED,
        "probability": False,
    }
    clf = SVC(**params)
    clf.fit(X_train, y_train_enc)
    y_pred = clf.predict(X_val)
    return accuracy_score(y_val_enc, y_pred)

study = optuna.create_study(
    direction="maximize",
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    sampler=TPESampler(multivariate=True),
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, n_jobs=OPTUNA_JOBS)

print("Best validation accuracy:", study.best_value)
print("Best params:", study.best_params)


e:\Nam_3_HK2\ThiGiac\DoAn\demo\.venv\Lib\site-packages\optuna\samplers\_tpe\sampler.py:319: ExperimentalWarning: ``multivariate`` option is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-05-14 11:27:52,841] A new study created in memory with name: no-name-25639011-51db-444e-9d47-26494dbbbd4b
[I 2026-05-14 11:28:36,158] Trial 1 finished with value: 0.810126582278481 and parameters: {'C': 0.6327393106255612, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 1 with value: 0.810126582278481.
[I 2026-05-14 11:28:40,129] Trial 0 finished with value: 0.24050632911392406 and parameters: {'C': 0.01528550356742748, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 1 with value: 0.810126582278481.
[I 2026-05-14 11:28:49,001] Trial 3 finished with value: 0.810126582278481 and parameters: {'C': 3.171620825960828, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 1 with value: 0.810126582278481.
[I 2026-05-14 11:28:53,713] Trial 2 finished with value: 

Best validation accuracy: 0.8396624472573839
Best params: {'C': 3.2252521973184405, 'kernel': 'rbf', 'gamma': 'scale'}


## Cell 6: Final Training & Evaluation

In [6]:

X_combined = np.vstack([X_train, X_val])
y_combined = np.concatenate([y_train_enc, y_val_enc])

best_params = dict(study.best_params)
best_params["random_state"] = SEED
best_params["probability"] = False

best_clf = SVC(**best_params)
best_clf.fit(X_combined, y_combined)

y_pred_test = best_clf.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test_enc, y_pred_test))
print(classification_report(y_test_enc, y_pred_test, target_names=le.classes_))



Test Accuracy: 0.7741935483870968
              precision    recall  f1-score   support

         Cam       0.85      0.92      0.88        36
      Chidan       0.74      0.69      0.71        36
    Hieulenh       0.68      0.68      0.68        31
    Nguyhiem       1.00      1.00      1.00        29
         Phu       0.55      0.52      0.53        23

    accuracy                           0.77       155
   macro avg       0.76      0.76      0.76       155
weighted avg       0.77      0.77      0.77       155



## Cell 7: Model Saving

In [7]:

model_path = MODELS_DIR / "HOG_SVM_8x2.joblib"
payload = {
    "model": best_clf,
    "label_encoder": le,
    "hog_params": HOG_PARAMS,
    "image_size": IMAGE_SIZE,
    "classes": list(le.classes_),
}
joblib.dump(payload, model_path)
print(f"Saved model to {model_path}")


Saved model to E:\Nam_3_HK2\ThiGiac\DoAn\demo\models\HOG_SVM_8x2.joblib
